# Perturbation operator catalogue

Every registered `company_perturbation` operator's effect on one name, across seeds, with the cleansed form of each result.

An operator declares where in a name it could act and how to make one change there; `company_perturbation.perturb` runs a chain of them under a seed. This notebook runs each registered operator alone, as a one-step chain, so what is shown is that operator and nothing else. It only picks a name and displays the result: nothing is read from or written to disk.

For which *scenarios* of a stored profile apply to a record, and what a whole chain does, use `notebooks/analyse_perturbation_profile.ipynb`.


In [ ]:
# --- parameters -------------------------------------------------------------
NAME = "Acme Systems Ltd"
COUNTRY = "gb"  # or None
SEEDS = range(5)  # the seeds drawn, one independent result each
PROBABILITY = 1.0  # the chance any one attempted change fires
ATTEMPTS = 1  # how many changes are tried, and so the most that may land
# ----------------------------------------------------------------------------

import polars as pl
from company_cleanse import strip_company_suffix
from company_perturbation import ChainStep, perturb, sited_registry

original_cleansed = strip_company_suffix(NAME, jurisdiction_code=COUNTRY)

rows = []
for operator_id in sited_registry.list_operators():
    operator = sited_registry.get(operator_id)
    for seed in SEEDS:
        result = perturb(
            NAME,
            [ChainStep(operator_id, PROBABILITY, ATTEMPTS)],
            seed=seed,
            country=COUNTRY,
        )
        mutated_cleansed = strip_company_suffix(result.name, jurisdiction_code=COUNTRY)
        rows.append(
            {
                "family": operator.family.value,
                "operator_id": operator_id,
                "seed": seed,
                "sites_offered": len(operator.sites(NAME, COUNTRY)),
                "changes_landed": len(result.changes),
                "mutated_name": result.name,
                "mutated_cleansed": mutated_cleansed,
                "absorbed_by_cleansing": mutated_cleansed == original_cleansed,
            }
        )

catalogue = pl.DataFrame(rows)
print(
    f"{catalogue.height} rows: {len(sited_registry.list_operators())} operator(s) "
    f"x {catalogue['seed'].n_unique()} seed(s); original cleansed form {original_cleansed!r}"
)
catalogue.sort(["family", "operator_id", "seed"])

## Per-operator summary

An operator draws from a random generator, and one draw does not characterise it, so this aggregates across every seed above.

`sites_offered` is how many places in this name the operator could act: zero means the name gives it nowhere to work, which is why `draws_changed` is zero for it, not a fault. `draws_absorbed_by_cleansing` counts the draws whose result `company_cleanse.strip_company_suffix` reduced back to the original's cleansed form: a perturbation cleansing absorbs cannot cost blocking recall, and one that survives is the only kind that can.


In [ ]:
summary = (
    catalogue.group_by(["family", "operator_id"])
    .agg(
        pl.col("sites_offered").first(),
        pl.len().alias("draws"),
        (pl.col("changes_landed") > 0).sum().alias("draws_changed"),
        pl.col("mutated_name").n_unique().alias("distinct_results"),
        pl.col("absorbed_by_cleansing").sum().alias("draws_absorbed_by_cleansing"),
    )
    .sort(["family", "operator_id"])
)
summary

## One operator, every draw

Pick a single operator id (from the `operator_id` column above) and see its result under every seed drawn.


In [ ]:
# --- pick one operator to inspect closely ------------------------------------
FOCUS_OPERATOR = min(catalogue["operator_id"].unique())
# ------------------------------------------------------------------------------

catalogue.filter(pl.col("operator_id") == FOCUS_OPERATOR).select(
    "seed",
    "changes_landed",
    "mutated_name",
    "mutated_cleansed",
    "absorbed_by_cleansing",
).sort("seed")